# ⛓️ Competitive Intelligence with Prompt Chaining
## Archetype 2 of 2 — Sequential LangGraph Chain

---

This notebook researches the **same product category** using a classic **prompt chaining** approach —  
a fixed sequence of LLM calls where each step's output is passed as context to the next.

### How prompt chaining works here

```
Step 1: Discover top N competitors           [~500 tokens in context]
  ↓ output appended
Step 2a: Research competitor 1 (Tavily × 3) [~2,500 tokens]
  ↓ output appended  
Step 2b: Research competitor 2 (Tavily × 3) [~5,000 tokens] ← growing fast
  ↓ output appended
Step 2c–2e: Research competitors 3–5        [~12,000 tokens] ← everyone's here now
  ↓ FULL context forwarded
Step 3: Sentiment analysis                  [12,000+ tokens as input]
  ↓ output appended
Step 4: Synthesize final brief              [15,000+ tokens as input]
  ↓ output appended
Step 5: Build recommendation matrix         [18,000+ tokens as input]
```

**Key insight:** Every step pays for every previous step's tokens again.  
Context scales linearly with depth — no isolation, no forgetting.

---

### Pipeline modeled as a LangGraph graph

We use LangGraph to represent the chain as a graph for:
- Consistent reporting (same metrics as the DeepAgents notebook)
- Visual clarity of the pipeline topology
- Step-by-step token tracking

> Compare final metrics to `01_deepagents_competitive_intel.ipynb`.

## 📦 1. Install Dependencies

In [ ]:
%pip install -q langchain langchain-openai langchain-community \
    langgraph langchain-core tavily-python langchain-tavily \
    python-dotenv pandas rich matplotlib ipywidgets

## 🔑 2. API Keys & Configuration

Use **the same settings** as the DeepAgents notebook for a fair comparison.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Override here if not using .env
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["TAVILY_API_KEY"] = "tvly-..."

assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"
assert os.environ.get("TAVILY_API_KEY"), "Missing TAVILY_API_KEY"

# ── Must match notebook 01 for a fair comparison ──────────────────────────
PRODUCT_CATEGORY    = "AI coding assistants"   # ← Same as notebook 01
NUM_COMPETITORS     = 5                          # ← Same as notebook 01
MODEL_NAME          = "gpt-5-mini"              # ← 500K TPM limit, same as notebook 01
MAX_SEARCH_RESULTS  = 5                          # ← Same as notebook 01
# ─────────────────────────────────────────────────────────────────────────

print(f"✅ Keys loaded | Category: {PRODUCT_CATEGORY} | Model: {MODEL_NAME}")

## 📊 3. LangGraph Reporting Setup

Same shared reporter as notebook 01.

In [ ]:
import sys
sys.path.insert(0, ".")

from utils.reporting import TokenCostTracker, build_report, compare_runs

tracker = TokenCostTracker(model_name=MODEL_NAME)
print("✅ LangGraph reporter ready")

## 🏗️ 4. Define the LangGraph Chain Pipeline

The pipeline is modeled as a LangGraph `StateGraph`.  
Each **node** is one step in the chain; the **state** carries all accumulated context forward.

```
discover_competitors
         ↓
research_competitors  (loops over each — context grows at every iteration)
         ↓
analyze_sentiment
         ↓
synthesize_brief
         ↓
build_matrix
         ↓
END
```

In [ ]:
import json
import time
from typing import Any, Dict, List, Optional, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, END
from IPython.display import Markdown, display

# ── LLM and tools ─────────────────────────────────────────────────────────
llm = ChatOpenAI(model=MODEL_NAME, temperature=0.1)

tavily = TavilySearch(
    max_results=MAX_SEARCH_RESULTS,
    include_answer=True,
    include_raw_content=False,
)

# ── State definition ──────────────────────────────────────────────────────
class ChainState(TypedDict):
    # Configuration
    product_category: str
    num_competitors: int
    # Pipeline outputs — each step appends to context
    competitors: List[str]                          # Step 1 output
    research_context: str                           # Grows after each research step
    competitor_research: Dict[str, str]             # Per-competitor summaries
    sentiment_analysis: str                         # Step 3 output
    final_brief: str                                # Step 4 output
    recommendation_matrix: str                      # Step 5 output
    # Tracking
    context_size_log: List[Dict[str, Any]]          # Logs context size at each step
    step_timings: List[Dict[str, float]]            # Wall-clock per step

print("✅ State schema defined")

### Node 1 — Discover Competitors
A single LLM call to identify the top N competitors.  
Starting context is small — this is where prompt chaining looks efficient.

In [ ]:
def discover_competitors(state: ChainState) -> ChainState:
    """Step 1: Ask the LLM to identify the top N competitors."""
    t0 = time.time()
    category = state["product_category"]
    n = state["num_competitors"]

    prompt = f"""List the top {n} companies/products competing in the '{category}' market as of 2025.
Return ONLY a JSON array of company/product names, nothing else.
Example: ["GitHub Copilot", "Cursor", "Tabnine", "Codeium", "Amazon Q"]"""

    response = llm.invoke([HumanMessage(content=prompt)], config={"callbacks": [tracker]})

    # Parse the JSON list
    try:
        raw = response.content.strip().strip("```json").strip("```").strip()
        competitors = json.loads(raw)
    except Exception:
        # Fallback: split by newlines if JSON parse fails
        competitors = [line.strip().strip('",-') for line in response.content.split("\n") if line.strip()]
        competitors = competitors[:n]

    elapsed = time.time() - t0
    print(f"\n🔍 [Step 1] Discovered {len(competitors)} competitors in {elapsed:.1f}s:")
    for i, c in enumerate(competitors, 1):
        print(f"   {i}. {c}")

    context_size = len(response.content)
    return {
        **state,
        "competitors": competitors,
        "research_context": f"# Competitors Identified\n{chr(10).join(f'{i+1}. {c}' for i, c in enumerate(competitors))}\n\n",
        "competitor_research": {},
        "context_size_log": [{"step": "discover_competitors", "context_chars": context_size, "elapsed_s": elapsed}],
        "step_timings": [{"step": "discover_competitors", "elapsed_s": elapsed}],
    }

print("✅ Node 1 defined: discover_competitors")

### Node 2 — Research Each Competitor
This is where the context **compounds**.  
Each competitor's research (Tavily search → LLM summarize) is appended to `research_context`.  
By the 5th competitor, we're passing the full accumulated context as input to every LLM call.

In [ ]:
def research_competitors(state: ChainState) -> ChainState:
    """
    Step 2: Research each competitor in sequence.
    IMPORTANT: research_context grows with each iteration.
    Every LLM call in this loop receives all previous research as context.
    """
    competitors = state["competitors"]
    research_context = state["research_context"]
    competitor_research = {}
    context_log = list(state.get("context_size_log", []))
    timings = list(state.get("step_timings", []))

    for i, competitor in enumerate(competitors, 1):
        t0 = time.time()
        print(f"\n🔬 [Step 2.{i}] Researching: {competitor}")
        print(f"   Context size before this step: {len(research_context):,} chars")

        # ── Tavily searches (3 per competitor) ──────────────────────────
        searches = [
            f"{competitor} pricing plans 2025",
            f"{competitor} new features announcements 2025",
            f"{competitor} user reviews Reddit OR HackerNews OR G2 2025",
        ]
        search_results = []
        for query in searches:
            try:
                result = tavily.invoke(query)
                tracker.tavily_searches += 1
                tracker.tool_calls += 1
                tracker.steps.append({"type": "tool", "name": "tavily_search", "input": query[:80]})
                # Extract text content
                if isinstance(result, list):
                    content = " ".join([r.get("content", "") for r in result[:3]])
                elif isinstance(result, dict):
                    content = result.get("answer", "") or " ".join(
                        [r.get("content", "") for r in result.get("results", [])[:3]]
                    )
                else:
                    content = str(result)
                search_results.append(f"Search: {query}\n{content[:800]}")
                print(f"   🌐 Searched: {query[:60]}...")
            except Exception as e:
                search_results.append(f"Search failed for: {query} — {e}")

        raw_search_text = "\n\n".join(search_results)

        # ── LLM call: summarize with ALL prior context in the prompt ─────
        # THIS IS THE CORE COST DRIVER: we pass research_context (growing!) 
        # plus the new search results into every summarization call.
        summarize_prompt = f"""You are a competitive intelligence analyst.

## Prior research context (accumulated):
{research_context}

## New search results for: {competitor}
{raw_search_text}

Summarize {competitor} covering:
1. Pricing & plans (include specific numbers if found)
2. Key features & differentiators
3. Recent announcements (last 6 months)
4. User sentiment & notable complaints
5. Target market & positioning

Be concise but complete. Use markdown formatting."""

        response = llm.invoke(
            [HumanMessage(content=summarize_prompt)],
            config={"callbacks": [tracker]},
        )
        summary = response.content
        competitor_research[competitor] = summary

        # ── Append to growing context ────────────────────────────────────
        research_context += f"\n\n## {competitor}\n{summary}"

        elapsed = time.time() - t0
        print(f"   Context size AFTER this step: {len(research_context):,} chars  (+{elapsed:.1f}s)")
        context_log.append({
            "step": f"research_{competitor}",
            "context_chars": len(research_context),
            "elapsed_s": elapsed,
        })
        timings.append({"step": f"research_{competitor}", "elapsed_s": elapsed})

    print(f"\n📦 Total research context size: {len(research_context):,} chars")

    return {
        **state,
        "competitor_research": competitor_research,
        "research_context": research_context,
        "context_size_log": context_log,
        "step_timings": timings,
    }

print("✅ Node 2 defined: research_competitors")

### Node 3 — Sentiment Analysis
Receives the **entire** accumulated research context as input.  
At this point you're paying for 5 competitors worth of tokens — again.

In [ ]:
def analyze_sentiment(state: ChainState) -> ChainState:
    """Step 3: Cross-competitor sentiment analysis using full accumulated context."""
    t0 = time.time()
    research_context = state["research_context"]

    print(f"\n💬 [Step 3] Sentiment analysis")
    print(f"   Prompt input size: {len(research_context):,} chars (this is the FULL context so far)")

    prompt = f"""Based on the following competitive research, provide a structured sentiment analysis \
comparing user satisfaction across all competitors. Include:
- Overall NPS proxy for each competitor (Positive/Mixed/Negative)
- Top 3 praise themes per competitor
- Top 3 complaint themes per competitor  
- Sentiment trends (improving / declining / stable)

## Full research context:
{research_context}

Format as a structured markdown report."""

    response = llm.invoke(
        [HumanMessage(content=prompt)],
        config={"callbacks": [tracker]},
    )
    elapsed = time.time() - t0

    print(f"   Completed in {elapsed:.1f}s")
    ctx_log = state.get("context_size_log", [])
    ctx_log.append({"step": "analyze_sentiment", "context_chars": len(research_context), "elapsed_s": elapsed})

    return {
        **state,
        "sentiment_analysis": response.content,
        "research_context": research_context + "\n\n## Sentiment Analysis\n" + response.content,
        "context_size_log": ctx_log,
    }

print("✅ Node 3 defined: analyze_sentiment")

### Node 4 — Synthesize the Final Brief
The most expensive LLM call: synthesizes everything from the maximal context window.

In [ ]:
def synthesize_brief(state: ChainState) -> ChainState:
    """Step 4: Synthesize the full competitive intelligence brief."""
    t0 = time.time()
    research_context = state["research_context"]

    print(f"\n📝 [Step 4] Synthesizing final brief")
    print(f"   Prompt input size: {len(research_context):,} chars")

    prompt = f"""You are a senior competitive intelligence analyst writing an executive brief.

Using the comprehensive research and sentiment analysis below, write a polished competitive \
intelligence brief with these sections:

1. **Executive Summary** (3-4 sentences)
2. **Competitor Profiles** (one subsection per competitor)
3. **Pricing Comparison Table** (markdown table)
4. **Sentiment Summary** (wins and risks per competitor)
5. **Strategic Recommendations** (numbered list, actionable)
6. **White Space Opportunities** (gaps in the market)

## Full research context:
{research_context}

Write in a professional analyst tone. Use markdown formatting throughout."""

    response = llm.invoke(
        [HumanMessage(content=prompt)],
        config={"callbacks": [tracker]},
    )
    elapsed = time.time() - t0

    print(f"   Completed in {elapsed:.1f}s")
    ctx_log = state.get("context_size_log", [])
    ctx_log.append({"step": "synthesize_brief", "context_chars": len(research_context), "elapsed_s": elapsed})

    return {
        **state,
        "final_brief": response.content,
        "research_context": research_context + "\n\n## Final Brief\n" + response.content,
        "context_size_log": ctx_log,
    }

print("✅ Node 4 defined: synthesize_brief")

### Node 5 — Build Recommendation Matrix
Final step — by now the context is at its maximum size.

In [ ]:
def build_matrix(state: ChainState) -> ChainState:
    """Step 5: Build a structured recommendation matrix."""
    t0 = time.time()
    research_context = state["research_context"]

    print(f"\n🎯 [Step 5] Building recommendation matrix")
    print(f"   Prompt input size: {len(research_context):,} chars (peak context)")

    prompt = f"""Based on all the competitive research and analysis below, create a concise \
recommendation matrix as a markdown table with these columns:

| Competitor | Best For | Avoid If | Threat Level (1-5) | Opportunity | Recommended Action |

Then add 3 prioritized strategic actions for a new entrant to this market.

## Full context:
{research_context}"""

    response = llm.invoke(
        [HumanMessage(content=prompt)],
        config={"callbacks": [tracker]},
    )
    elapsed = time.time() - t0

    print(f"   Completed in {elapsed:.1f}s")
    ctx_log = state.get("context_size_log", [])
    ctx_log.append({"step": "build_matrix", "context_chars": len(research_context), "elapsed_s": elapsed})

    return {
        **state,
        "recommendation_matrix": response.content,
        "context_size_log": ctx_log,
    }

print("✅ Node 5 defined: build_matrix")

### Compile the LangGraph Pipeline

In [ ]:
from langgraph.graph import StateGraph, END

builder = StateGraph(ChainState)

builder.add_node("discover_competitors",  discover_competitors)
builder.add_node("research_competitors",  research_competitors)
builder.add_node("analyze_sentiment",     analyze_sentiment)
builder.add_node("synthesize_brief",      synthesize_brief)
builder.add_node("build_matrix",          build_matrix)

builder.set_entry_point("discover_competitors")
builder.add_edge("discover_competitors", "research_competitors")
builder.add_edge("research_competitors", "analyze_sentiment")
builder.add_edge("analyze_sentiment",    "synthesize_brief")
builder.add_edge("synthesize_brief",     "build_matrix")
builder.add_edge("build_matrix",         END)

chain_graph = builder.compile()
print("✅ LangGraph prompt chain compiled")
print(f"   Nodes: {list(builder.nodes.keys())}")

## 🚀 5. Run the Chain

Watch the **context size** printed at each step — this is the core story.  
Compare it to the sub-agent isolation in notebook 01.

> ⏱️ This will take several minutes.

In [ ]:
import time

tracker = TokenCostTracker(model_name=MODEL_NAME)  # fresh tracker
start_time = time.time()

initial_state: ChainState = {
    "product_category": PRODUCT_CATEGORY,
    "num_competitors": NUM_COMPETITORS,
    "competitors": [],
    "research_context": "",
    "competitor_research": {},
    "sentiment_analysis": "",
    "final_brief": "",
    "recommendation_matrix": "",
    "context_size_log": [],
    "step_timings": [],
}

print(f"⛓️  Starting Prompt Chain pipeline...")
print(f"   Category: {PRODUCT_CATEGORY}")
print(f"   Model:    {MODEL_NAME}\n")
print("━" * 60)

final_state = chain_graph.invoke(initial_state)

tracker.stop()
total_elapsed = time.time() - start_time
print("\n" + "━" * 60)
print(f"✅ Chain complete in {total_elapsed:.1f}s")

## 📄 6. Final Intelligence Brief

In [ ]:
display(Markdown(final_state["final_brief"]))

## 🎯 7. Recommendation Matrix

In [ ]:
display(Markdown(final_state["recommendation_matrix"]))

## 📊 8. LangGraph Cost & Token Report

In [ ]:
report = build_report(
    tracker=tracker,
    approach="Prompt Chaining",
    result_text=final_state["final_brief"],
)

prompt_chaining_summary = tracker.summary()
print("\n📋 Summary dict (use in comparison notebook):")
print(json.dumps(prompt_chaining_summary, indent=2))

## 📈 9. Context Growth Visualization

This chart shows the defining characteristic of prompt chaining:  
**context size grows monotonically at every step.** Every LLM call pays for all prior work.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

ctx_log = final_state.get("context_size_log", [])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Prompt Chaining — Token Usage\n{PRODUCT_CATEGORY}", fontsize=14, fontweight="bold")

# ── Left: Context growth over pipeline steps ───────────────────────────
ax1 = axes[0]
if ctx_log:
    steps = [c["step"].replace("research_", "📊 ").replace("_", " ") for c in ctx_log]
    ctx_sizes = [c["context_chars"] for c in ctx_log]
    colors = ["#E25A4A" if "research" in c["step"] else "#4A90E2" for c in ctx_log]
    bars = ax1.barh(steps, ctx_sizes, color=colors)
    ax1.set_xlabel("Context Size (chars)")
    ax1.set_title("Context Growth at Each Step")
    ax1.invert_yaxis()
    # Annotate bars
    for bar, val in zip(bars, ctx_sizes):
        ax1.text(val + 100, bar.get_y() + bar.get_height()/2,
                 f"{val:,}", va="center", fontsize=8)
    red_patch = mpatches.Patch(color="#E25A4A", label="Research steps (compounding)")
    blue_patch = mpatches.Patch(color="#4A90E2", label="Analysis steps")
    ax1.legend(handles=[red_patch, blue_patch], fontsize=8)
else:
    ax1.text(0.5, 0.5, "No context log data", ha="center", va="center")
    ax1.set_title("Context Growth at Each Step")

# ── Right: Token pie ────────────────────────────────────────────────────
ax2 = axes[1]
labels = ["Prompt Tokens", "Completion Tokens"]
sizes = [tracker.prompt_tokens, tracker.completion_tokens]
colors2 = ["#4A90E2", "#E25A4A"]
if sum(sizes) > 0:
    ax2.pie(sizes, labels=labels, colors=colors2, autopct="%1.1f%%", startangle=90)
else:
    ax2.text(0.5, 0.5, "No token data\n(run cell 5 first)", ha="center", va="center")
ax2.set_title("Token Split")

plt.tight_layout()
plt.savefig("prompt_chaining_tokens.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved to prompt_chaining_tokens.png")

## ⚖️ 10. Side-by-Side Comparison

Paste the `deepagents_summary` dict from notebook 01 into the cell below  
to generate the side-by-side comparison table.

In [ ]:
# ── Paste the summary from notebook 01 here ──────────────────────────────
# Run notebook 01 first, then copy the printed JSON into this dict.
deepagents_summary = {
    "model": "gpt-4o",
    "llm_calls": 0,           # ← replace with actual value from notebook 01
    "tool_calls": 0,
    "tavily_searches": 0,
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
    "elapsed_seconds": 0.0,
}

compare_runs(deepagents_summary, prompt_chaining_summary)

## 📈 11. Context Growth: DeepAgents vs Prompt Chaining

This chart makes the architectural difference **visually unmistakable** for your blog post.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate the theoretical context growth pattern
# In DeepAgents: orchestrator context stays flat (sub-agents are isolated)
# In Prompt Chaining: context grows at every step

n = NUM_COMPETITORS
steps = ["Plan"] + [f"Competitor {i}" for i in range(1, n+1)] + ["Sentiment", "Synthesis", "Matrix"]
total_steps = len(steps)

# Estimate based on actual data from the run
if ctx_log and tracker.total_tokens > 0:
    # Use actual data from this run
    pc_ctx = [0] + [c["context_chars"] for c in ctx_log]
    # DeepAgents: sub-agents are ~equal size, orchestrator never accumulates
    avg_sub_agent_ctx = (pc_ctx[n] // n) if n > 0 else 1000
    da_ctx = [avg_sub_agent_ctx] * total_steps  # roughly fixed isolation
else:
    # Illustrative example data
    pc_ctx = [500, 2500, 5000, 8000, 11000, 14000, 17000, 20000, 23000]
    da_ctx = [1500] * total_steps  # sub-agent context windows are isolated

# Normalize lengths
min_len = min(len(steps), len(pc_ctx), len(da_ctx))
steps, pc_ctx, da_ctx = steps[:min_len], pc_ctx[:min_len], da_ctx[:min_len]

fig, ax = plt.subplots(figsize=(13, 5))
x = range(len(steps))
ax.plot(x, pc_ctx, "o-", color="#E25A4A", linewidth=2.5, markersize=8, label="Prompt Chaining (context grows)")
ax.plot(x, da_ctx, "s--", color="#27AE60", linewidth=2.5, markersize=8, label="DeepAgents (sub-agents isolated)")
ax.fill_between(x, da_ctx, pc_ctx, alpha=0.12, color="#E25A4A", label="Token cost premium of chaining")
ax.set_xticks(list(x))
ax.set_xticklabels(steps, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Context Size (chars)")
ax.set_title(
    f"Context Size Growth: DeepAgents vs Prompt Chaining\n{PRODUCT_CATEGORY}",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("context_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved to context_comparison.png — great for your blog post!")

## ✅ 12. Summary

| What Prompt Chaining did | Trade-off |
|---|---|
| Fixed sequence of steps modeled as LangGraph nodes | Transparent, easy to reason about, debuggable |
| Context accumulated at every step | Token cost scales as O(n²) with depth × breadth |
| Every LLM call re-received all prior research | No isolation — data from competitor 1 biases competitor 5 research |
| Tavily called explicitly, per step | No parallelism — all research is sequential |
| Works well for short, focused tasks | Breaks down for multi-entity, multi-step research |

---

### When Prompt Chaining wins
- Short, linear tasks (summarize → translate → format)
- Tasks where context continuity matters (story generation, code refactoring)
- When you need full control over every step
- When cost is less important than simplicity

### When DeepAgents wins
- Multi-entity research (5 competitors, 10 candidates, 20 products)
- Long-horizon tasks that need planning and replanning
- Tasks that benefit from parallel isolated execution
- When you need file-based state persistence and resumability